Гафутулина Эльвира 6403-010302D

#Лабораторная работа №1

1. Найти велосипед с максимальным временем пробега.
2. Найти наибольшее геодезическое расстояние между станциями.
3. Найти путь велосипеда с максимальным временем пробега через станции.
4. Найти количество велосипедов в системе.
5. Найти пользователей потративших на поездки более 3 часов.

In [ ]:
from pyspark import SparkContext, SparkConf
from pyspark.sql import SparkSession
from pyspark.sql.types import DoubleType
import pyspark.sql as sql
import pyspark.sql.functions as F
from math import sqrt
import pyspark.sql.types as t
from geopy.distance import geodesic

In [ ]:
conf = SparkConf().setAppName("L1_Apache_Spark").setMaster('local[*]')

In [ ]:
sc = SparkContext(conf=conf)
spark = SparkSession(sc)

In [ ]:
trips = spark.read.format('csv').option('header', 'true').load("trips.csv")

In [ ]:
trips.show(1, vertical=True)

-RECORD 0----------------------------------
 id                 | 4576                 
 duration           | 63                   
 start_date         | NULL                 
 start_station_name | South Van Ness at... 
 start_station_id   | 66                   
 end_date           | 8/29/2013 14:14      
 end_station_name   | South Van Ness at... 
 end_station_id     | 66                   
 bike_id            | 520                  
 subscription_type  | Subscriber           
 zip_code           | 94127                
only showing top 1 row


In [ ]:
stations = spark.read.format('csv').option('header', 'true').load("stations.csv")

In [ ]:
stations.show(1, vertical=True)

-RECORD 0---------------------------------
 id                | 2                    
 name              | San Jose Diridon ... 
 lat               | 37.329732            
 long              | -121.90178200000001  
 dock_count        | 27                   
 city              | San Jose             
 installation_date | 8/6/2013             
only showing top 1 row


In [ ]:
trips = (
    trips
    .where(F.col("id").isNotNull())
    .where(F.col("duration").isNotNull())
    .where(F.col("bike_id").isNotNull())
    .cache()
)

stations = (
    stations
    .where(F.col("id").isNotNull())
    .where(F.col("lat").isNotNull())
    .where(F.col("long").isNotNull())
    .cache()
)

print(f"Количество строк в trips: {trips.count()}")
print(f"Количество строк в stations: {stations.count()}")

Количество строк в trips: 669958
Количество строк в stations: 70


##1. Найти велосипед с максимальным временем пробега.

In [ ]:
Maxtime = (
    trips
    .groupBy('bike_id')
    .agg(
        F.max(F.col("duration").cast(t.IntegerType())).alias("duration")
    )
)
Maxtime = Maxtime.orderBy(F.col('duration').desc())
Maxtime.show(1)

+-------+--------+
|bike_id|duration|
+-------+--------+
|    535|17270400|
+-------+--------+
only showing top 1 row


##2. Найти наибольшее геодезическое расстояние между станциями.

In [ ]:
stations_clean = (
    stations
    .select(
        F.col("id").alias("id"),
        F.col("lat").cast(DoubleType()).alias("lat"),
        F.col("long").cast(DoubleType()).alias("lon")
    )
)

pairs = (
    stations_clean.alias("a")
    .join(stations_clean.alias("b"), F.col("a.id") < F.col("b.id"))
)

def geo_distance(lat1, lon1, lat2, lon2):
    return float(geodesic((lat1, lon1), (lat2, lon2)).km)

geo_udf = F.udf(geo_distance, DoubleType())

distances = pairs.withColumn(
    "distance_km",
    geo_udf(
        F.col("a.lat"), F.col("a.lon"),
        F.col("b.lat"), F.col("b.lon")
    )
)

result = (
    distances
    .orderBy(F.col("distance_km").desc())
    .select(
        F.col("a.id").alias("station_1"),
        F.col("b.id").alias("station_2"),
        "distance_km"
    )
)

result.show(1, truncate=False)

+---------+---------+-----------------+
|station_1|station_2|distance_km      |
+---------+---------+-----------------+
|16       |60       |69.92096757764355|
+---------+---------+-----------------+
only showing top 1 row


## 3. Найти путь велосипеда с максимальным временем пробега через станции.

In [ ]:
bike_max = (
    trips
    .withColumn("duration", F.col("duration").cast("int"))
    .groupBy("bike_id")
    .agg(F.sum("duration").alias("total_time"))
    .orderBy(F.desc("total_time"))
    .first()
)

bike_id_max = bike_max["bike_id"]

print(f"Велосипед: {bike_id_max}")


bike_trips = (
    trips
    .select(
        F.col("id").cast("int").alias("trip_id"),
        "bike_id",
        "start_station_name",
        "end_station_name"
    )
    .filter(F.col("bike_id") == bike_id_max)
    .orderBy("trip_id")
)

print("\nПоездки велосипеда:")
bike_trips.show(bike_trips.count(), truncate=False)


rows = bike_trips.collect()

route = []

for i in range(len(rows)):
    if i == 0:
        route.append(rows[i]["start_station_name"])
    route.append(rows[i]["end_station_name"])


print("\nПоследовательность станций:")
print(" -> ".join(route))

Велосипед: 535

Поездки велосипеда:
+-------+-------+---------------------------------------------+---------------------------------------------+
|trip_id|bike_id|start_station_name                           |end_station_name                             |
+-------+-------+---------------------------------------------+---------------------------------------------+
|4966   |535    |Post at Kearney                              |San Francisco Caltrain (Townsend at 4th)     |
|5067   |535    |San Francisco Caltrain (Townsend at 4th)     |San Francisco Caltrain 2 (330 Townsend)      |
|5179   |535    |San Francisco Caltrain 2 (330 Townsend)      |Market at Sansome                            |
|5199   |535    |Market at Sansome                            |2nd at South Park                            |
|7806   |535    |2nd at Townsend                              |Davis at Jackson                             |
|11422  |535    |San Francisco City Hall                      |Civic Center BART (7t

##4. Найти количество велосипедов в системе.

In [ ]:
bike_count = trips.select("bike_id").distinct().count()

print(f"Количество велосипедов в системе: {bike_count}")

Количество велосипедов в системе: 700


##5. Найти пользователей потративших на поездки более 3 часов.

In [ ]:
trips_clean = trips.withColumn("duration", F.col("duration").cast("int")).filter(F.col("zip_code").isNotNull())

users_time = (
    trips_clean
    .groupBy("zip_code")
    .agg(F.sum("duration").alias("total_time"))
)

result = users_time.filter(F.col("total_time") > 10800)
result = result.orderBy(F.col("total_time").desc())

print("Пользователи с суммарным временем > 3 часов:")
result.show(truncate=False)

Пользователи с суммарным временем > 3 часов:
+--------+----------+
|zip_code|total_time|
+--------+----------+
|94107   |49757162  |
|nil     |45725550  |
|94105   |25596128  |
|94133   |21637675  |
|94102   |19128021  |
|94103   |19127388  |
|95531   |17270400  |
|94111   |14244997  |
|95112   |12742370  |
|94109   |12057128  |
|94040   |7807926   |
|94110   |7421936   |
|94117   |6901313   |
|94301   |6590378   |
|94041   |6276284   |
|94158   |6248167   |
|94306   |5550643   |
|94025   |5178237   |
|94108   |5127562   |
|94611   |5014906   |
+--------+----------+
only showing top 20 rows
